<a href="https://colab.research.google.com/github/IraSamsonova/nn/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random


In [2]:
# Выбор устройства: GPU если доступен, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
# Читаем текстовый файл целиком
with open("/content/tinyshakespeare.txt", encoding="utf-8") as f:
    text = f.read()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# все уникальные символы
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Словари для кодирования и декодирования
stoi = {ch: i for i, ch in enumerate(chars)}  # символ в индекс
itos = {i: ch for i, ch in enumerate(chars)}  # индекс в символ

In [6]:
# кодирование строки в тензор индексов
def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)


In [7]:
# Делим текст на обучающую и валидационную части
split = int(0.9 * len(text))
train_text = text[:split]
val_text = text[split:]


In [8]:
class CharRNN(nn.Module): # RNN-модель для символьной генерации текста

    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()

        # Преобразует индекс символа в вектор
        self.embed = nn.Embedding(vocab_size, embed_size)

        # RNN слой
        self.rnn = nn.RNN(
            embed_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Линейный слой для предсказания следующего символа
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):

        # x: [batch_size, seq_len] → [batch_size, seq_len, embed_size]
        x = self.embed(x)

        # out: [batch_size, seq_len, hidden_size]
        out, hidden = self.rnn(x, hidden)

        # Преобразуем скрытое состояние в логиты [batch_size, seq_len, vocab_size]
        out = self.fc(out)

        return out, hidden

    def init_hidden(self, batch_size): # Инициализация скрытого состояния нулями

        return torch.zeros(
            num_layers,
            batch_size,
            hidden_size
        ).to(device)


In [11]:
embed_size = 32
hidden_size = 128 # Размер скрытого состояния RNN
num_layers = 2
seq_len = 64 # Длина входной последовательности
batch_size = 64
epochs = 20
lr = 0.003


In [12]:
def get_batches(data, batch_size, seq_len):
    max_start = len(data) - seq_len - 1 # Максимальный возможный стартовый индекс
    starts = torch.randint(0, max_start, (batch_size,)) # Случайные стартовые позиции для каждого элемента батча

    x = torch.stack([encode(data[s:s+seq_len]) for s in starts])
    y = torch.stack([encode(data[s+1:s+seq_len+1]) for s in starts])

    return x.to(device), y.to(device)


In [13]:
model = CharRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss() # Функция потерь для классификации символов

steps_per_epoch = 1000


In [14]:
for epoch in range(epochs):
    model.train()
    hidden = model.init_hidden(batch_size)
    train_loss = 0

    for _ in range(steps_per_epoch):
        x, y = get_batches(train_text, batch_size, seq_len)

        optimizer.zero_grad()
        out, hidden = model(x, hidden.detach())
        loss = criterion(out.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()


    model.eval()
    val_loss = 0
    with torch.no_grad():
        x, y = get_batches(val_text, batch_size, seq_len)
        out, _ = model(x, None)
        val_loss = criterion(
            out.view(-1, vocab_size),
            y.view(-1)
        ).item()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train loss: {train_loss/steps_per_epoch:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )


Epoch 1/20 | Train loss: 1.8100 | Val loss: 1.7571
Epoch 2/20 | Train loss: 1.5544 | Val loss: 1.7129
Epoch 3/20 | Train loss: 1.5111 | Val loss: 1.6855
Epoch 4/20 | Train loss: 1.4912 | Val loss: 1.6064
Epoch 5/20 | Train loss: 1.4792 | Val loss: 1.7032
Epoch 6/20 | Train loss: 1.4722 | Val loss: 1.6316
Epoch 7/20 | Train loss: 1.4638 | Val loss: 1.6483
Epoch 8/20 | Train loss: 1.4622 | Val loss: 1.6645
Epoch 9/20 | Train loss: 1.4562 | Val loss: 1.6456
Epoch 10/20 | Train loss: 1.4545 | Val loss: 1.5971
Epoch 11/20 | Train loss: 1.4516 | Val loss: 1.6883
Epoch 12/20 | Train loss: 1.4495 | Val loss: 1.6881
Epoch 13/20 | Train loss: 1.4479 | Val loss: 1.6249
Epoch 14/20 | Train loss: 1.4455 | Val loss: 1.6878
Epoch 15/20 | Train loss: 1.4432 | Val loss: 1.6543
Epoch 16/20 | Train loss: 1.4434 | Val loss: 1.6454
Epoch 17/20 | Train loss: 1.4425 | Val loss: 1.6738
Epoch 18/20 | Train loss: 1.4422 | Val loss: 1.6571
Epoch 19/20 | Train loss: 1.4404 | Val loss: 1.6265
Epoch 20/20 | Train l

In [15]:
# посимвольно генерирует текст
def generate(model, start="KING ", length=300, temperature=0.8):

    model.eval()

    # Кодируем начальную строку
    x = encode(start).unsqueeze(0).to(device)

    # Инициализируем скрытое состояние
    hidden = model.init_hidden(1)

    result = start

    for _ in range(length):
        # Прямой проход
        out, hidden = model(x, hidden)

        # Берём логиты последнего символа
        logits = out[:, -1] / temperature

        # Преобразуем в вероятности
        probs = torch.softmax(logits, dim=-1)

        # Сэмплируем следующий символ
        idx = torch.multinomial(probs, 1).item()

        # Добавляем символ к результату
        result += itos[idx]

        # Следующий вход — только что сгенерированный символ
        x = torch.tensor([[idx]]).to(device)

    print(result)


In [16]:

generate(model, "KING RICHARD: ")


KING RICHARD: for you will be Flew with that though show to the feast with part the people: what lease who should besoming her of my bleep.

JULIET:
For any wise and loves.

VIRGILIA:
As I would hat well.

EXTON:
No, for the ware kind his honour of the king like a mighty mouths.
Till give my mapery are a man may 
